# T-3 Forecasting Benchmark Report


In [8]:
from cybench.evaluation.eval import implemented_metrics
from cybench.util.config_utils import get_run_description
import pathlib
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import os
from cybench.config import (
    DATASETS,
    PATH_DATA_DIR,
    PATH_RESULTS_DIR,
    KEY_COUNTRY,
    KEY_LOC,
    KEY_YEAR,
    KEY_TARGET,
)

def calculate_metrics_for_run(df: pd.DataFrame, run_name: str):
    """Calculates all defined metrics for a single run DataFrame."""
    if 'targets' not in df.columns or 'preds' not in df.columns:
        return {}

    y_true = df['targets'].values
    y_pred = df['preds'].values
    mask = (~np.isnan(y_true)) & (~np.isnan(y_pred))
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    results = {}
    for name, func in implemented_metrics.items():
        try:
            results[name] = np.round(func(y_true, y_pred), 4)
        except Exception:
            results[name] = np.nan
    return results

# --- Main Analysis Function ---

def analyze_experiments(exp_path: str):
    """Loads all runs, calculates metrics, and outputs a styled table."""
    exp_dir = pathlib.Path(exp_path)
    if not exp_dir.exists():
        return f"Error: Experiment path not found: {exp_path}"

    all_results = []

    for run_dir in exp_dir.iterdir():
        if run_dir.is_dir() and run_dir.name != '.hydra':

            # Find preds.csv recursively to handle nested structures
            preds_file = next(run_dir.glob('**/preds.csv'), None)
            overrides_path = run_dir / '.hydra' / 'overrides.yaml'

            if preds_file and overrides_path.exists():
                run_description = get_run_description(overrides_path)

                try:
                    df_run = pd.read_csv(preds_file)
                    metrics_dict = calculate_metrics_for_run(df_run, run_description)

                    if metrics_dict:
                        metrics_dict['Run Description'] = run_description
                        all_results.append(metrics_dict)
                except Exception as e:
                    print(f"Error processing {run_dir.name}: {e}")

    if not all_results:
        return "No successful runs were processed or data files were found."

    # Create the final results DataFrame
    final_df = pd.DataFrame(all_results).set_index('Run Description')

    # Ensure 'r2' is the first column if it exists
    if 'r2' in final_df.columns:
        cols = ['r2'] + [col for col in final_df.columns if col != 'r2']
        final_df = final_df[cols]

    # Style the DataFrame: RdBu colormap from -1 to 1 for R2
    styled_df = final_df.style.background_gradient(
        subset=['r2'],
        cmap='RdBu',
        vmin=-1,
        vmax=1
    ).format(
        precision=4 # Format all numeric columns
    ).set_caption("Experiment Evaluation Summary")

    return styled_df

In [21]:
exp_path = "../cybench/output/cnn_lstm_US_small"

analyze_experiments(exp_path)

,r2,mse,normalized_rmse,mape,r
Run Description,,,,,
model.epochs=50 | model.torch_model.embed_dim=64 | model=cnn_lf,0.6519,2.0527,13.9534,0.1350,0.8286
model.epochs=100 | model.torch_model.embed_dim=64 | model=cnn_lf,0.5904,2.4151,15.1351,0.1417,0.8297
model.epochs=50 | model.torch_model.embed_dim=128 | model=cnn_lf,0.7236,1.6297,12.4331,0.1171,0.8525
model.epochs=100 | model.torch_model.embed_dim=128 | model=cnn_lf,0.6571,2.0220,13.8487,0.1318,0.8141
model.epochs=50 | model.torch_model.embed_dim=64 | model=lstm_lf,0.6959,1.7927,13.0401,0.1200,0.8344
model.epochs=100 | model.torch_model.embed_dim=64 | model=lstm_lf,0.6722,1.9331,13.5408,0.1270,0.8304
model.epochs=50 | model.torch_model.embed_dim=128 | model=lstm_lf,0.7200,1.6511,12.5144,0.1162,0.8488
model.epochs=100 | model.torch_model.embed_dim=128 | model=lstm_lf,0.6979,1.7814,12.9987,0.1206,0.8470
